# Diagnostic d'intégrité des TIF d'inférence

Ce notebook scanne les fichiers `.tif` produits par l'inférence et identifie les fichiers corrompus ou illisibles.

**Structure attendue :**
```
<base_path>/
├── mode_produit/<timestamp>/*.tif
├── mode_consecutif/<timestamp>/*.tif   (à venir)
└── mode_aleatoire/<timestamp>/*.tif    (à venir)
```

**Vérifications effectuées par fichier :**
1. Taille du fichier > seuil minimal
2. Ouverture rasterio réussie
3. Dimensions non nulles
4. Nombre de bandes > 0 et multiple de 12
5. Lecture échantillon (bande 1 et dernière bande)
6. Lecture complète de toutes les bandes (détection corruption partielle)
7. Détection de bandes entièrement à zéro
8. CRS présent
9. dtype = uint16

## 1. Imports et configuration

In [ ]:
import re
import warnings
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

plt.rcParams.update({"savefig.dpi": 300, "figure.dpi": 150})

## 2. Paramètres

Modifier les paramètres ci-dessous selon votre configuration.

In [ ]:
BASE_PATH = Path("/mnt/stores/store_dai/tmp/speillet/inferences/v3_combined")

# Modes d'inférence attendus (un répertoire par mode)
MODES = ["mode_produit", "mode_consecutif", "mode_aleatoire"]

# Nombre de workers pour le scan parallèle
N_WORKERS = 4

# Taille minimale d'un fichier .tif valide (en Ko)
MIN_FILE_SIZE_KB = 10

# Nombre de bandes Sentinel-2 par date
BANDS_PER_DATE = 12

# Regex pour parser les noms de fichiers
FILENAME_PATTERN = re.compile(r"pred_mgrsc_([A-Z0-9]+)_row-(\d+)_col-(\d+)\.tif")

## 3. Fonction de vérification d'un fichier TIF

In [ ]:
def check_single_tif(filepath: Path) -> dict:
    """Vérifie l'intégrité complète d'un fichier TIFF.

    Retourne un dict avec le statut ('ok' ou 'error'), les erreurs,
    les warnings et les métadonnées du fichier.
    """
    result = {
        "filepath": str(filepath),
        "filename": filepath.name,
        "status": "ok",
        "errors": [],
        "warnings": [],
        "file_size_mb": 0.0,
        "width": None,
        "height": None,
        "n_bands": None,
        "dtype": None,
        "crs": None,
        "driver": None,
        "compress": None,
        "zero_bands_count": 0,
        "zero_bands_indices": [],
    }

    # --- Vérification taille fichier ---
    try:
        size_bytes = filepath.stat().st_size
        result["file_size_mb"] = round(size_bytes / (1024 * 1024), 2)
        if size_bytes < MIN_FILE_SIZE_KB * 1024:
            result["status"] = "error"
            result["errors"].append(
                f"Fichier trop petit : {result['file_size_mb']:.2f} Mo "
                f"(seuil : {MIN_FILE_SIZE_KB} Ko)"
            )
            return result
    except OSError as e:
        result["status"] = "error"
        result["errors"].append(f"Impossible de lire le fichier : {e}")
        return result

    # --- Ouverture rasterio + vérifications ---
    try:
        with rasterio.open(filepath) as ds:
            result["width"] = ds.width
            result["height"] = ds.height
            result["n_bands"] = ds.count
            result["dtype"] = str(ds.dtypes[0]) if ds.dtypes else None
            result["crs"] = str(ds.crs) if ds.crs else None
            result["driver"] = ds.profile.get("driver", "unknown")
            result["compress"] = ds.profile.get("compress", None)

            # Dimensions nulles
            if ds.width == 0 or ds.height == 0:
                result["status"] = "error"
                result["errors"].append(
                    f"Dimensions nulles : {ds.width}x{ds.height}"
                )

            # Nombre de bandes
            if ds.count == 0:
                result["status"] = "error"
                result["errors"].append("Aucune bande trouvée")
                return result
            elif ds.count % BANDS_PER_DATE != 0:
                result["warnings"].append(
                    f"Nombre de bandes ({ds.count}) n'est pas un "
                    f"multiple de {BANDS_PER_DATE}"
                )

            # dtype
            if ds.dtypes and ds.dtypes[0] != "uint16":
                result["warnings"].append(
                    f"dtype inattendu : {ds.dtypes[0]} (attendu : uint16)"
                )

            # CRS
            if ds.crs is None:
                result["warnings"].append("CRS absent")

            # --- Lecture complète bande par bande ---
            zero_bands = []
            for band_idx in range(1, ds.count + 1):
                try:
                    data = ds.read(band_idx)
                    if data is None:
                        result["status"] = "error"
                        result["errors"].append(
                            f"Bande {band_idx} : read() retourne None"
                        )
                    elif np.all(data == 0):
                        zero_bands.append(band_idx)
                except Exception as e:
                    result["status"] = "error"
                    result["errors"].append(
                        f"Bande {band_idx}/{ds.count} : erreur lecture - {e}"
                    )

            result["zero_bands_count"] = len(zero_bands)
            result["zero_bands_indices"] = zero_bands
            if zero_bands:
                result["warnings"].append(
                    f"{len(zero_bands)} bande(s) entièrement à zéro : "
                    f"{zero_bands[:10]}{'...' if len(zero_bands) > 10 else ''}"
                )

    except rasterio.errors.RasterioIOError as e:
        result["status"] = "error"
        result["errors"].append(f"RasterioIOError : {e}")
    except OSError as e:
        result["status"] = "error"
        result["errors"].append(f"OSError : {e}")
    except Exception as e:
        result["status"] = "error"
        result["errors"].append(f"{type(e).__name__} : {e}")

    return result

## 4. Scan de tous les modes

In [ ]:
def scan_all_modes(
    base_path: Path, modes: list[str], workers: int = 4
) -> list[dict]:
    """Scanne tous les modes disponibles et vérifie chaque .tif."""
    all_results = []

    for mode in modes:
        mode_dir = base_path / mode
        if not mode_dir.exists():
            print(f"⏭  {mode} : répertoire absent, ignoré")
            continue

        tif_files = sorted(mode_dir.rglob("*.tif"))
        if not tif_files:
            print(f"⚠  {mode} : aucun fichier .tif trouvé")
            continue

        print(f"🔍 {mode} : {len(tif_files)} fichier(s) .tif")

        if workers <= 1:
            for fp in tqdm(tif_files, desc=mode):
                r = check_single_tif(fp)
                r["mode"] = mode
                all_results.append(r)
        else:
            with ProcessPoolExecutor(max_workers=workers) as executor:
                futures = {
                    executor.submit(check_single_tif, fp): fp for fp in tif_files
                }
                for future in tqdm(
                    as_completed(futures), total=len(futures), desc=mode
                ):
                    r = future.result()
                    r["mode"] = mode
                    all_results.append(r)

    return all_results

## 5. Exécution du scan

In [ ]:
results = scan_all_modes(BASE_PATH, MODES, workers=N_WORKERS)
print(f"\nTotal : {len(results)} fichier(s) analysé(s)")

## 6. Construction du DataFrame récapitulatif

In [ ]:
def build_dataframe(results: list[dict]) -> pd.DataFrame:
    """Construit un DataFrame à partir des résultats du scan."""
    rows = []
    for r in results:
        m = FILENAME_PATTERN.match(r["filename"])
        tile_id = m.group(1) if m else None
        row_idx = int(m.group(2)) if m else None
        col_idx = int(m.group(3)) if m else None

        parent_name = Path(r["filepath"]).parent.name

        rows.append({
            "mode": r["mode"],
            "timestamp": parent_name,
            "filename": r["filename"],
            "tile_id": tile_id,
            "row": row_idx,
            "col": col_idx,
            "status": r["status"],
            "n_errors": len(r["errors"]),
            "errors": "; ".join(r["errors"]) if r["errors"] else "",
            "n_warnings": len(r["warnings"]),
            "warnings": "; ".join(r["warnings"]) if r["warnings"] else "",
            "width": r["width"],
            "height": r["height"],
            "n_bands": r["n_bands"],
            "dtype": r["dtype"],
            "crs": r["crs"],
            "file_size_mb": r["file_size_mb"],
            "driver": r["driver"],
            "compress": r["compress"],
            "zero_bands_count": r["zero_bands_count"],
            "filepath": r["filepath"],
        })

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(["mode", "tile_id", "row", "col"]).reset_index(drop=True)
    return df


df = build_dataframe(results)
print(f"{len(df)} fichier(s) dans le DataFrame")
df.head()

## 7. Résumé global par mode

In [ ]:
def style_summary(row):
    """Colorie les cellules du résumé."""
    styles = [""] * len(row)
    if row["erreurs"] > 0:
        styles[row.index.get_loc("erreurs")] = "background-color: #ffcccc"
    if row["warnings"] > 0:
        styles[row.index.get_loc("warnings")] = "background-color: #fff3cd"
    if row["erreurs"] == 0 and row["warnings"] == 0:
        styles[row.index.get_loc("ok")] = "background-color: #d4edda"
    return styles


if not df.empty:
    summary = (
        df.groupby("mode")
        .agg(
            fichiers=("filename", "count"),
            erreurs=("n_errors", lambda x: (x > 0).sum()),
            warnings=("n_warnings", lambda x: (x > 0).sum()),
            ok=("status", lambda x: (x == "ok").sum()),
            taille_totale_mb=("file_size_mb", "sum"),
        )
        .reset_index()
    )
    display(summary.style.apply(style_summary, axis=1).format({"taille_totale_mb": "{:.1f}"}))
else:
    print("Aucun fichier à analyser.")

## 8. Détail des fichiers en erreur

In [ ]:
df_errors = df[df["status"] == "error"] if not df.empty else pd.DataFrame()

if df_errors.empty:
    print("✅ Aucune erreur détectée sur l'ensemble des fichiers.")
else:
    print(f"❌ {len(df_errors)} fichier(s) en erreur :\n")
    for _, row in df_errors.iterrows():
        print(f"  [{row['mode']}] {row['filename']}")
        print(f"    Taille : {row['file_size_mb']:.2f} Mo | "
              f"Bandes : {row['n_bands']} | "
              f"Dimensions : {row['width']}x{row['height']}")
        for err in row["errors"].split("; "):
            if err:
                print(f"    ❌ {err}")
        print()

    display(
        df_errors[
            ["mode", "filename", "tile_id", "row", "col",
             "file_size_mb", "n_bands", "errors"]
        ].style.set_properties(**{"background-color": "#ffcccc"})
    )

## 9. Détail des fichiers avec warnings

In [ ]:
df_warnings = df[(df["n_warnings"] > 0) & (df["status"] == "ok")] if not df.empty else pd.DataFrame()

if df_warnings.empty:
    print("✅ Aucun warning détecté sur les fichiers valides.")
else:
    print(f"⚠️  {len(df_warnings)} fichier(s) avec des warnings :\n")
    for _, row in df_warnings.iterrows():
        print(f"  [{row['mode']}] {row['filename']}")
        for w in row["warnings"].split("; "):
            if w:
                print(f"    ⚠️  {w}")
        print()

    display(
        df_warnings[
            ["mode", "filename", "tile_id", "n_bands",
             "zero_bands_count", "warnings"]
        ].style.set_properties(**{"background-color": "#fff3cd"})
    )

## 10. Cohérence inter-modes

Vérifie que les différents modes contiennent le même nombre et les mêmes fichiers (même set de tuiles).
Fonctionne avec un seul mode ou avec tous les modes.

In [ ]:
if df.empty:
    print("Aucun fichier à comparer.")
else:
    modes_present = df["mode"].unique().tolist()
    print(f"Modes détectés : {modes_present}\n")

    if len(modes_present) < 2:
        print(f"Un seul mode disponible ({modes_present[0]}), "
              "la comparaison inter-modes sera possible quand les autres "
              "modes seront générés.")
    else:
        counts = df.groupby("mode")["filename"].count()
        print("Nombre de fichiers par mode :")
        print(counts.to_string())
        print()

        if counts.nunique() != 1:
            print("⚠️  Le nombre de fichiers diffère entre les modes !\n")

        filenames_by_mode = {
            mode: set(grp["filename"].values)
            for mode, grp in df.groupby("mode")
        }
        all_filenames = set().union(*filenames_by_mode.values())

        missing_data = []
        for mode in modes_present:
            missing = all_filenames - filenames_by_mode[mode]
            if missing:
                for fn in sorted(missing):
                    missing_data.append({"mode": mode, "fichier_manquant": fn})

        if missing_data:
            print("❌ Fichiers manquants d'un mode à l'autre :\n")
            display(pd.DataFrame(missing_data))
        else:
            print("✅ Tous les modes ont exactement les mêmes fichiers.")

## 11. Cohérence des métadonnées

Vérifie que tous les fichiers partagent les mêmes propriétés (dtype, CRS, dimensions, nombre de bandes).

In [ ]:
if not df.empty:
    for col in ["dtype", "crs", "width", "height", "driver", "compress"]:
        unique_vals = df[col].dropna().unique()
        if len(unique_vals) == 1:
            print(f"✅ {col} : uniforme ({unique_vals[0]})")
        elif len(unique_vals) == 0:
            print(f"⚠️  {col} : aucune valeur")
        else:
            print(f"❌ {col} : valeurs hétérogènes ({', '.join(map(str, unique_vals))})")
            display(df.groupby(col)["filename"].count().rename("nb_fichiers"))

    print("\n--- Distribution du nombre de bandes ---")
    band_counts = df["n_bands"].dropna().value_counts().sort_index()
    print(band_counts.to_string())

    if band_counts.shape[0] > 1:
        fig, ax = plt.subplots(figsize=(8, 3))
        band_counts.plot.bar(ax=ax, color="steelblue", edgecolor="black")
        ax.set_xlabel("Nombre de bandes")
        ax.set_ylabel("Nombre de fichiers")
        ax.set_title("Distribution du nombre de bandes par fichier")
        plt.tight_layout()
        plt.show()
else:
    print("Aucun fichier à analyser.")

## 12. Visualisations

In [ ]:
if not df.empty and df["tile_id"].notna().any():
    # --- Bar chart : erreurs et warnings par tile_id ---
    tile_stats = (
        df.groupby("tile_id")
        .agg(
            erreurs=("n_errors", lambda x: (x > 0).sum()),
            warnings=("n_warnings", lambda x: (x > 0).sum()),
            total=("filename", "count"),
        )
        .sort_index()
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    colors = ["#e74c3c" if v > 0 else "#2ecc71" for v in tile_stats["erreurs"]]
    tile_stats["erreurs"].plot.bar(ax=axes[0], color=colors, edgecolor="black")
    axes[0].set_title("Erreurs par tuile MGRS")
    axes[0].set_ylabel("Nombre de fichiers en erreur")
    axes[0].tick_params(axis="x", rotation=45)

    colors_w = ["#f39c12" if v > 0 else "#2ecc71" for v in tile_stats["warnings"]]
    tile_stats["warnings"].plot.bar(ax=axes[1], color=colors_w, edgecolor="black")
    axes[1].set_title("Warnings par tuile MGRS")
    axes[1].set_ylabel("Nombre de fichiers avec warnings")
    axes[1].tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.show()

    # --- Heatmap : statut par (tile_id, row_col) ---
    df_plot = df.copy()
    df_plot["row_col"] = "r" + df_plot["row"].astype(str) + "_c" + df_plot["col"].astype(str)
    df_plot["status_code"] = df_plot["status"].map({"ok": 0, "error": 1})
    df_plot.loc[
        (df_plot["status"] == "ok") & (df_plot["n_warnings"] > 0), "status_code"
    ] = 0.5

    for mode in df_plot["mode"].unique():
        df_mode = df_plot[df_plot["mode"] == mode]
        pivot = df_mode.pivot_table(
            index="tile_id", columns="row_col",
            values="status_code", aggfunc="max"
        )
        if pivot.empty:
            continue

        fig, ax = plt.subplots(
            figsize=(max(6, len(pivot.columns) * 1.2), max(4, len(pivot.index) * 0.5))
        )
        from matplotlib.colors import ListedColormap
        cmap = ListedColormap(["#2ecc71", "#f39c12", "#e74c3c"])
        im = ax.imshow(pivot.values, cmap=cmap, vmin=0, vmax=1, aspect="auto")
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels(pivot.columns, rotation=45, ha="right")
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels(pivot.index)
        ax.set_title(f"Statut par tuile — {mode}")

        from matplotlib.patches import Patch
        legend_elements = [
            Patch(facecolor="#2ecc71", label="OK"),
            Patch(facecolor="#f39c12", label="Warning"),
            Patch(facecolor="#e74c3c", label="Erreur"),
        ]
        ax.legend(handles=legend_elements, loc="upper right", bbox_to_anchor=(1.25, 1))

        plt.tight_layout()
        plt.show()
else:
    print("Pas de données à visualiser.")

## 13. Export CSV (optionnel)

Décommenter la cellule ci-dessous pour sauvegarder le rapport en CSV.

In [ ]:
# csv_path = BASE_PATH / "rapport_integrite_tif.csv"
# df.to_csv(csv_path, index=False)
# print(f"Rapport exporté : {csv_path}")